# Práctica 4 – Sistemas Inteligentes Aplicados a la Salud  
### Darío Meneses
## Ejercicio 1: Ronda clínica simulada (A* + Algoritmos Genéticos con DEAP)

**Objetivo:** encontrar el mejor orden de visita a las habitaciones indicadas, partiendo del despacho **D20** y terminando en **UCI1**, minimizando el **coste total del recorrido**.

El coste entre dos puntos se calcula como el camino óptimo en el plano del hospital mediante **A\***, considerando las **penalizaciones** del mapa.


In [7]:
# =========================
# IMPORTS 
# =========================

import warnings
warnings.filterwarnings("ignore")

import sys
import types
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from deap import base, creator, tools

# -------------------------
# 1) Importamos el MAPA y ELITISMO desde utils
# -------------------------
from utils.mapa_hospital import mapa_hospital, penalizaciones
from utils.elitism import eaSimpleWithElitism

# -------------------------
# 2) Importamos Nodo y Abiertos desde clases/clases.py
#    
# -------------------------
from clases.clases import Nodo, Abiertos
mod_abiertos = types.ModuleType("clases.abiertos")
mod_abiertos.Abiertos = Abiertos
sys.modules["clases.abiertos"] = mod_abiertos


from utils.busquedaAestrella import busqueda_A_estrella


from clases.clasesVisualizacion import VisualizadorHospital, SolucionesPosiciones

print("Imports OK ")


Imports OK 


In [8]:
# =========================
# DATOS DEL PROBLEMA
# =========================

START = "D20"
END   = "UCI1"

VISITS = ["1S","4S","7S","10S","14S","17S","2N","4N","5N","12N","19N"]

ALL_POINTS = [START] + VISITS + [END]

print("Número total de puntos (incluyendo inicio/fin):", len(ALL_POINTS))
ALL_POINTS


Número total de puntos (incluyendo inicio/fin): 13


['D20',
 '1S',
 '4S',
 '7S',
 '10S',
 '14S',
 '17S',
 '2N',
 '4N',
 '5N',
 '12N',
 '19N',
 'UCI1']

In [ ]:
# ============================================================
# Localización de los puntos del problema en el mapa
# ============================================================

def find_positions(label, grid):
    """
    Esta función permite localizar todas las posiciones del mapa del hospital
    en las que aparece una determinada etiqueta (habitaciones, despacho, UCI, etc.).

    Args:
        label (str):
            Etiqueta que se quiere localizar en el mapa del hospital
            (por ejemplo 'D20', 'UCI1', '1S', ...).

        grid (list of list):
            Mapa del hospital representado como una matriz de etiquetas.

    Returns:
        list of tuple:
            Lista de coordenadas (fila, columna) donde aparece la etiqueta.
            Si la etiqueta no está presente en el mapa, la lista estará vacía.
    """
    positions = []

    # Recorremos el mapa completo buscando la etiqueta
    for i, row in enumerate(grid):
        for j, cell in enumerate(row):
            if str(cell) == label:
                positions.append((i, j))

    return positions


# ------------------------------------------------------------
# Asociación de cada punto del problema con una posición
# concreta del mapa del hospital
# ------------------------------------------------------------
# En el mapa del hospital algunas zonas (habitaciones, UCI,
# despacho, etc.) ocupan varias celdas. Para simplificar el
# problema de búsqueda, se selecciona una única celda
# representativa para cada punto, concretamente la primera
# que aparece en el recorrido del mapa.
#
# Esta aproximación es suficiente para el objetivo del
# ejercicio, ya que el coste real del desplazamiento se
# calcula posteriormente mediante el algoritmo A*, teniendo
# en cuenta las penalizaciones del mapa.
# ------------------------------------------------------------

POINT_POS = {}

for point in ALL_POINTS:
    pos_list = find_positions(point, mapa_hospital)

    # Comprobación de seguridad: si una etiqueta no existe,
    # el problema no estaría bien definido
    if not pos_list:
        raise ValueError(f"No se encuentra la etiqueta {point} en el mapa.")

    # Se toma la primera aparición como posición representativa
    POINT_POS[point] = pos_list[0]


# Se muestran las posiciones obtenidas, lo cual resulta útil
# tanto para depuración como para la explicación en la memoria
POINT_POS


{'D20': (28, 64),
 '1S': (33, 7),
 '4S': (28, 11),
 '7S': (33, 21),
 '10S': (28, 25),
 '14S': (28, 43),
 '17S': (33, 53),
 '2N': (6, 7),
 '4N': (6, 11),
 '5N': (1, 15),
 '12N': (6, 39),
 '19N': (1, 57),
 'UCI1': (9, 38)}

In [10]:
# ============================================================
# Definición del problema para la búsqueda A*
# ============================================================

# Movimientos permitidos en el mapa del hospital:
# arriba, abajo, izquierda y derecha
MOVIMIENTOS = [(-1, 0), (1, 0), (0, -1), (0, 1)]


def dentro_del_mapa(pos):
    """
    Comprueba si una posición está dentro de los límites del mapa del hospital.

    Args:
        pos (tuple):
            Coordenada (fila, columna) a comprobar.

    Returns:
        bool:
            True si la posición es válida dentro del mapa, False en caso contrario.
    """
    i, j = pos
    return 0 <= i < len(mapa_hospital) and 0 <= j < len(mapa_hospital[0])


def es_muro(pos):
    """
    Indica si una posición corresponde a un muro del hospital.

    Args:
        pos (tuple):
            Coordenada (fila, columna) a evaluar.

    Returns:
        bool:
            True si la celda es un muro ('M'), False en caso contrario.
    """
    i, j = pos
    return str(mapa_hospital[i][j]) == "M"


def coste_celda(pos):
    """
    Devuelve el coste de atravesar una celda del mapa del hospital.

    El coste depende del tipo de zona y se obtiene del diccionario de
    penalizaciones proporcionado en la práctica.

    Args:
        pos (tuple):
            Coordenada (fila, columna) de la celda.

    Returns:
        int:
            Coste asociado a dicha celda.
    """
    i, j = pos
    return penalizaciones.get(str(mapa_hospital[i][j]), 1)


class ProblemaHospital:
    """
    Clase que modela el problema de búsqueda en el hospital para el algoritmo A*.
    """

    def __init__(self, posicion_inicial, posicion_objetivo):
        """
        Inicializa el problema de búsqueda A*.

        Args:
            posicion_inicial (tuple):
                Coordenada inicial (fila, columna).

            posicion_objetivo (tuple):
                Coordenada objetivo (fila, columna).
        """
        self.posicion_inicial = posicion_inicial
        self.posicion_objetivo = posicion_objetivo

        # Nodo inicial con coste g = 0 y heurística h calculada
        self.inicial = Nodo(
            posicion_inicial,
            None,
            g=0,
            h=self.heuristica(posicion_inicial)
        )

    def heuristica(self, pos):
        """
        Heurística utilizada por el algoritmo A*.

        Se emplea la distancia Manhattan, adecuada para mapas en cuadrícula
        y admisible al no sobreestimar el coste real.

        Args:
            pos (tuple):
                Posición actual (fila, columna).

        Returns:
            int:
                Valor heurístico estimado hasta el objetivo.
        """
        return abs(pos[0] - self.posicion_objetivo[0]) + abs(pos[1] - self.posicion_objetivo[1])

    def es_objetivo(self, nodo):
        """
        Comprueba si el nodo corresponde al estado objetivo.

        Args:
            nodo (Nodo):
                Nodo a comprobar.

        Returns:
            bool:
                True si el nodo es el objetivo, False en caso contrario.
        """
        return nodo.getEstado() == self.posicion_objetivo

    def sucesores(self, nodo):
        """
        Genera los sucesores válidos de un nodo dado.

        Args:
            nodo (Nodo):
                Nodo actual.

        Returns:
            list[Nodo]:
                Lista de nodos sucesores válidos.
        """
        sucesores = []
        i, j = nodo.getEstado()

        for di, dj in MOVIMIENTOS:
            nueva_pos = (i + di, j + dj)

            # Comprobaciones básicas
            if not dentro_del_mapa(nueva_pos):
                continue
            if es_muro(nueva_pos):
                continue

            # Cálculo del nuevo coste acumulado
            g_nuevo = nodo.getG() + coste_celda(nueva_pos)
            h_nuevo = self.heuristica(nueva_pos)

            sucesores.append(
                Nodo(nueva_pos, nodo, g=g_nuevo, h=h_nuevo)
            )

        return sucesores


In [12]:
# ============================================================
# Preprocesado con A*: matriz de costes y caminos
# ============================================================
# Idea:
#   - El algoritmo genético va a probar MUCHOS órdenes distintos.
#   - Si cada vez que evalúo un individuo tuviese que ejecutar A*, sería lentísimo.
#   - Por eso, antes de lanzar el genético, calculo una matriz COST[a][b] con el
#     coste mínimo entre cada par de puntos (a -> b) usando A*.
#   - También guardo PATHS[a][b] con el camino (lista de coordenadas) para luego
#     poder visualizar la ruta final en el mapa del hospital.
# ============================================================

from functools import lru_cache

def camino_y_coste_entre(etiqueta_origen, etiqueta_destino):
    """
    Calcula el camino y el coste mínimo entre dos puntos del hospital utilizando A*.

    Args:
        etiqueta_origen (str):
            Etiqueta del punto de origen (ej. 'D20', '1S', ...)

        etiqueta_destino (str):
            Etiqueta del punto destino (ej. 'UCI1', '4N', ...)

    Returns:
        tuple:
            (camino, coste)
            - camino: lista de coordenadas (fila, col) desde origen hasta destino
            - coste : coste total acumulado del camino (según penalizaciones)
            Si no existiera solución, devuelve (None, infinito).
    """
    origen = POINT_POS[etiqueta_origen]
    destino = POINT_POS[etiqueta_destino]

    problema = ProblemaHospital(origen, destino)

    # traza=False para que no imprima información extra
    solucion = busqueda_A_estrella(problema, traza=False)

    if solucion is None:
        return None, float("inf")

    return solucion.camino(), solucion.getG()


# ------------------------------------------------------------
# Construimos:
#   COST[a][b]  -> coste mínimo entre a y b
#   PATHS[a][b] -> lista de posiciones para el camino entre a y b
# ------------------------------------------------------------

COST  = {a: {} for a in ALL_POINTS}
PATHS = {a: {} for a in ALL_POINTS}

t0 = time.perf_counter()

for a in ALL_POINTS:
    for b in ALL_POINTS:

        # Caso trivial: mismo punto
        if a == b:
            COST[a][b] = 0
            PATHS[a][b] = [POINT_POS[a]]
            continue

        camino, coste = camino_y_coste_entre(a, b)

        COST[a][b] = coste
        PATHS[a][b] = camino


t1 = time.perf_counter()
print(f"Matriz de costes ")

# Mostramos la matriz 
pd.DataFrame(COST).loc[ALL_POINTS, ALL_POINTS]


Matriz de costes 


,D20,1S,4S,7S,10S,14S,17S,2N,4N,5N,12N,19N,UCI1
D20,0,360,504,444,392,374,314,387,383,527,469,533,2854
1S,260,0,209,169,121,139,99,116,112,256,218,282,2603
4S,404,209,0,313,265,283,243,260,256,400,362,426,2747
7S,344,169,313,0,205,223,183,208,204,348,302,366,2687
10S,292,121,265,205,0,171,131,160,156,300,250,314,2635
14S,274,139,283,223,171,0,113,170,166,310,252,316,2637
17S,214,99,243,183,131,113,0,126,122,266,208,272,2593
2N,287,116,260,208,160,170,126,0,57,213,187,255,2588
4N,283,112,256,204,156,166,122,57,0,209,183,251,2584
5N,427,256,400,348,300,310,266,213,209,0,327,395,2728


In [13]:
# ============================================================
# Coste de una ruta completa y reconstrucción del camino
# ============================================================

def coste_ruta(orden_visitas):
    """
    Calcula el coste total de una ruta completa.

    La ruta siempre comienza en el despacho inicial (D20),
    visita las habitaciones en el orden indicado y finaliza
    en la UCI (UCI1).

    Args:
        orden_visitas (list of str):
            Lista de etiquetas de las habitaciones en el orden
            en el que se visitan.

    Returns:
        float:
            Coste total de la ruta.
    """
    # Secuencia completa: inicio -> visitas -> fin
    secuencia = [START] + list(orden_visitas) + [END]

    coste_total = 0

    # Sumamos el coste entre puntos consecutivos
    for origen, destino in zip(secuencia[:-1], secuencia[1:]):
        coste_total += COST[origen][destino]

    return coste_total


def construir_camino_completo(orden_visitas):
    """
    Reconstruye el camino completo del médico dentro del hospital
    a partir de un orden de visitas.

    Para cada par consecutivo de puntos se concatena el camino
    calculado previamente con A* durante el preprocesado.

    Args:
        orden_visitas (list of str):
            Lista de habitaciones en el orden de visita.

    Returns:
        list of tuple:
            Lista de coordenadas (fila, columna) que representan
            el recorrido completo dentro del hospital.
    """
    secuencia = [START] + list(orden_visitas) + [END]
    camino_completo = []

    for origen, destino in zip(secuencia[:-1], secuencia[1:]):
        tramo = PATHS[origen][destino]

        # Evitamos duplicar la celda de unión entre tramos
        if camino_completo:
            camino_completo.extend(tramo[1:])
        else:
            camino_completo.extend(tramo)

    return camino_completo


In [14]:
print("Coste ejemplo D20 -> 1S -> UCI1:",
      coste_ruta(["1S"]))


Coste ejemplo D20 -> 1S -> UCI1: 5313
